# K-Means Clustering Assignment Solutions

This notebook solves all 5 problem statements from `7 K-Means Clustering (1).pdf`.

## Imports

In [ ]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## Utility Functions

In [ ]:
def pick_kmeans_k(x, k_min=2, k_max=10, random_state=42):
    ks = list(range(k_min, k_max + 1))
    inertia, sil = [], []
    for k in ks:
        km = KMeans(n_clusters=k, random_state=random_state, n_init=10)
        labels = km.fit_predict(x)
        inertia.append(float(km.inertia_))
        sil.append(float(silhouette_score(x, labels)))
    best_k = ks[int(np.argmax(sil))]
    return ks, inertia, sil, best_k

def cluster_summary(x, k):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km_labels = km.fit_predict(x)
    hc = AgglomerativeClustering(n_clusters=k, linkage='ward')
    hc_labels = hc.fit_predict(x)
    return {
        'kmeans_silhouette': silhouette_score(x, km_labels),
        'hierarchical_silhouette': silhouette_score(x, hc_labels),
        'ari_kmeans_vs_hierarchical': adjusted_rand_score(km_labels, hc_labels),
        'kmeans_cluster_sizes': pd.Series(km_labels).value_counts().sort_index().to_dict(),
        'hierarchical_cluster_sizes': pd.Series(hc_labels).value_counts().sort_index().to_dict(),
        'kmeans_labels': km_labels
    }

## Problem 1: Airlines Dataset

In [ ]:
air = pd.read_excel('EastWestAirlines (1) (1).xlsx', sheet_name='data')
x_air = air.drop(columns=['ID#'])
sc = StandardScaler()
x_air_s = sc.fit_transform(x_air)
ks, inertia, sil, best_k = pick_kmeans_k(x_air_s)
summary_air = cluster_summary(x_air_s, best_k)
print('Best K:', best_k)
print('Silhouette:', summary_air['kmeans_silhouette'])
print('Cluster sizes:', summary_air['kmeans_cluster_sizes'])
air.assign(cluster=summary_air['kmeans_labels']).groupby('cluster').mean(numeric_only=True).round(2)

## Problem 2: Crime Dataset

In [ ]:
crime = pd.read_excel('crime_data (1).xlsx')
x_crime = crime.drop(columns=['Unnamed: 0'])
x_crime_s = StandardScaler().fit_transform(x_crime)
ks, inertia, sil, best_k = pick_kmeans_k(x_crime_s)
summary_crime = cluster_summary(x_crime_s, best_k)
print('Best K:', best_k)
print('Silhouette:', summary_crime['kmeans_silhouette'])
crime.assign(cluster=summary_crime['kmeans_labels']).groupby('cluster').mean(numeric_only=True).round(2)

## Problem 3: Insurance Dataset

In [ ]:
ins = pd.read_excel('Insurance Dataset.xlsx')
x_ins_s = StandardScaler().fit_transform(ins)
ks, inertia, sil, best_k = pick_kmeans_k(x_ins_s)
summary_ins = cluster_summary(x_ins_s, best_k)
print('Best K:', best_k)
print('Silhouette:', summary_ins['kmeans_silhouette'])
ins.assign(cluster=summary_ins['kmeans_labels']).groupby('cluster').mean(numeric_only=True).round(2)

## Problem 4: Telecom Mixed Dataset

In [ ]:
tel = pd.read_excel('Telco_customer_churn (1) (1).xlsx')
tel = tel.drop(columns=['Customer ID', 'Count'])
num_cols = tel.select_dtypes(include=['number']).columns.tolist()
cat_cols = [c for c in tel.columns if c not in num_cols]
pre = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_cols),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_cols),
])
x_tel = pre.fit_transform(tel)
ks, inertia, sil, best_k = pick_kmeans_k(x_tel)
summary_tel = cluster_summary(x_tel, best_k)
print('Best K:', best_k)
print('Silhouette:', summary_tel['kmeans_silhouette'])
tmp = pd.read_excel('Telco_customer_churn (1) (1).xlsx')
tmp['cluster'] = summary_tel['kmeans_labels']
tmp.groupby('cluster')[['Tenure in Months','Monthly Charge','Total Charges','Total Revenue']].mean().round(2)

## Problem 5: AutoInsurance Mixed Dataset

In [ ]:
auto = pd.read_excel('AutoInsurance (1).xlsx')
auto['Effective To Date'] = pd.to_datetime(auto['Effective To Date'], errors='coerce').map(lambda x: x.toordinal() if pd.notna(x) else np.nan)
auto = auto.drop(columns=['Customer'])
num_cols = auto.select_dtypes(include=['number']).columns.tolist()
cat_cols = [c for c in auto.columns if c not in num_cols]
pre = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_cols),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_cols),
])
x_auto = pre.fit_transform(auto)
ks, inertia, sil, best_k = pick_kmeans_k(x_auto)
summary_auto = cluster_summary(x_auto, best_k)
print('Best K:', best_k)
print('Silhouette:', summary_auto['kmeans_silhouette'])
tmp2 = pd.read_excel('AutoInsurance (1).xlsx')
tmp2['cluster'] = summary_auto['kmeans_labels']
tmp2.groupby('cluster')[['Income','Customer Lifetime Value','Total Claim Amount']].mean().round(2)